# 🐘 Feature Extraction Exploration

## Purpose
Interactive exploration of the dual-branch feature extractor for elephant re-identification.

**Objectives:**
1. Load and test the dual-branch model
2. Visualize features from both branches
3. Analyze attention weights
4. Test on sample preprocessed images
5. Compare features across different elephant types

---

## 1. Setup & Dependencies

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import random
from typing import List, Tuple

# Import our models
from src.models.dual_branch_extractor import DualBranchFeatureExtractor
from src.models.texture_branch import TextureBranch
from src.models.semantic_branch import SemanticBranch

# Configure matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['figure.dpi'] = 100

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print("✓ Dependencies loaded")

## 2. Load Model

In [ ]:
# Initialize dual-branch model
model = DualBranchFeatureExtractor(
    input_channels=3,
    texture_dim=256,
    semantic_dim=256,
    fusion_dim=512,
    use_attention=True
).to(device)

# Set to evaluation mode
model.eval()

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Model on device: {next(model.parameters()).device}")

## 3. Load Sample Images

In [ ]:
# Paths to processed data
DATA_ROOT = project_root / "data" / "processed"
MAKHNA_DIR = DATA_ROOT / "Makhna"
HERD_DIR = DATA_ROOT / "Herd"

def load_sample_images(directory: Path, num_samples: int = 5) -> List[Path]:
    """Load random sample images from directory."""
    valid_extensions = {'.jpg', '.jpeg', '.JPG', '.JPEG', '.png', '.PNG'}
    images = []
    
    for root, dirs, files in os.walk(directory):
        for file in files:
            if Path(file).suffix in valid_extensions:
                images.append(Path(root) / file)
    
    return random.sample(images, min(num_samples, len(images)))

# Load samples
makhna_samples = load_sample_images(MAKHNA_DIR, num_samples=5)
herd_samples = load_sample_images(HERD_DIR, num_samples=5)

print(f"Loaded {len(makhna_samples)} Makhna samples")
print(f"Loaded {len(herd_samples)} Herd samples")
print("\nSample images:")
for i, img_path in enumerate(makhna_samples[:3], 1):
    print(f"  Makhna {i}: {img_path.name}")
for i, img_path in enumerate(herd_samples[:3], 1):
    print(f"  Herd {i}: {img_path.name}")

## 4. Image Preprocessing Utilities

In [ ]:
def preprocess_image(image_path: Path, target_size: Tuple[int, int] = (224, 224)) -> torch.Tensor:
    """
    Load and preprocess image for model input.
    
    Args:
        image_path: Path to image file
        target_size: Target size (height, width)
        
    Returns:
        Preprocessed tensor [1, 3, H, W]
    """
    # Load image
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Resize
    image = cv2.resize(image, target_size)
    
    # Normalize to [0, 1]
    image = image.astype(np.float32) / 255.0
    
    # Convert to tensor [C, H, W]
    image_tensor = torch.from_numpy(image).permute(2, 0, 1)
    
    # Add batch dimension [1, C, H, W]
    image_tensor = image_tensor.unsqueeze(0)
    
    return image_tensor

def display_image(image_path: Path, title: str = ""):
    """Display image with title."""
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(8, 6))
    plt.imshow(image)
    plt.title(title if title else image_path.name, fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("✓ Preprocessing utilities defined")

## 5. Extract Features from Sample Images

In [ ]:
# Extract features from one sample
sample_image = makhna_samples[0]
print(f"Extracting features from: {sample_image.name}")

# Display image
display_image(sample_image, f"Sample: {sample_image.name}")

# Preprocess
image_tensor = preprocess_image(sample_image).to(device)
print(f"Input tensor shape: {image_tensor.shape}")

# Extract features
with torch.no_grad():
    fused, texture, semantic, attention = model(
        image_tensor, 
        return_branch_features=True
    )

print(f"\nFeature shapes:")
print(f"  Fused features: {fused.shape}")
print(f"  Texture features: {texture.shape}")
print(f"  Semantic features: {semantic.shape}")
print(f"  Attention weights: {attention.shape}")

print(f"\nAttention weights (Texture/Semantic): {attention[0].cpu().numpy()}")
print(f"Feature norms:")
print(f"  Fused: {torch.norm(fused, p=2, dim=1).item():.4f}")
print(f"  Texture: {torch.norm(texture, p=2, dim=1).item():.4f}")
print(f"  Semantic: {torch.norm(semantic, p=2, dim=1).item():.4f}")

## 6. Visualize Attention Weights Across Samples

In [ ]:
# Extract features from all samples
def extract_features_batch(image_paths: List[Path]):
    """Extract features from batch of images."""
    features_list = []
    
    for img_path in image_paths:
        img_tensor = preprocess_image(img_path).to(device)
        
        with torch.no_grad():
            fused, texture, semantic, attention = model(
                img_tensor,
                return_branch_features=True
            )
        
        features_list.append({
            'name': img_path.name,
            'fused': fused.cpu().numpy(),
            'texture': texture.cpu().numpy(),
            'semantic': semantic.cpu().numpy(),
            'attention': attention.cpu().numpy()
        })
    
    return features_list

# Extract features
print("Extracting features from Makhna samples...")
makhna_features = extract_features_batch(makhna_samples)

print("Extracting features from Herd samples...")
herd_features = extract_features_batch(herd_samples)

print(f"\n✓ Extracted features from {len(makhna_features) + len(herd_features)} images")

In [ ]:
# Visualize attention weights
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Makhna attention weights
makhna_texture_weights = [f['attention'][0, 0] for f in makhna_features]
makhna_semantic_weights = [f['attention'][0, 1] for f in makhna_features]

x = np.arange(len(makhna_features))
width = 0.35

ax1.bar(x - width/2, makhna_texture_weights, width, label='Texture', color='skyblue')
ax1.bar(x + width/2, makhna_semantic_weights, width, label='Semantic', color='coral')
ax1.set_xlabel('Sample Index', fontsize=12)
ax1.set_ylabel('Attention Weight', fontsize=12)
ax1.set_title('Makhna: Branch Attention Weights', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Herd attention weights
herd_texture_weights = [f['attention'][0, 0] for f in herd_features]
herd_semantic_weights = [f['attention'][0, 1] for f in herd_features]

x = np.arange(len(herd_features))

ax2.bar(x - width/2, herd_texture_weights, width, label='Texture', color='skyblue')
ax2.bar(x + width/2, herd_semantic_weights, width, label='Semantic', color='coral')
ax2.set_xlabel('Sample Index', fontsize=12)
ax2.set_ylabel('Attention Weight', fontsize=12)
ax2.set_title('Herd: Branch Attention Weights', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics
print("\nAttention Weight Statistics:")
print("=" * 60)
print(f"Makhna (Adult males):")
print(f"  Avg Texture:  {np.mean(makhna_texture_weights):.3f}")
print(f"  Avg Semantic: {np.mean(makhna_semantic_weights):.3f}")
print(f"\nHerd (Females/Juveniles):")
print(f"  Avg Texture:  {np.mean(herd_texture_weights):.3f}")
print(f"  Avg Semantic: {np.mean(herd_semantic_weights):.3f}")

## 7. Feature Similarity Analysis

In [ ]:
def cosine_similarity(feat1: np.ndarray, feat2: np.ndarray) -> float:
    """Calculate cosine similarity between two feature vectors."""
    return np.dot(feat1.flatten(), feat2.flatten())

# Calculate similarity matrix for Makhna samples
n_makhna = len(makhna_features)
similarity_matrix = np.zeros((n_makhna, n_makhna))

for i in range(n_makhna):
    for j in range(n_makhna):
        sim = cosine_similarity(
            makhna_features[i]['fused'],
            makhna_features[j]['fused']
        )
        similarity_matrix[i, j] = sim

# Visualize similarity matrix
plt.figure(figsize=(10, 8))
plt.imshow(similarity_matrix, cmap='viridis', aspect='auto')
plt.colorbar(label='Cosine Similarity')
plt.title('Feature Similarity Matrix (Makhna Samples)', fontsize=14, fontweight='bold')
plt.xlabel('Sample Index', fontsize=12)
plt.ylabel('Sample Index', fontsize=12)
plt.tight_layout()
plt.show()

print("Similarity matrix shape:", similarity_matrix.shape)
print(f"Diagonal (self-similarity): {np.diag(similarity_matrix)}")
print(f"Mean off-diagonal similarity: {(similarity_matrix.sum() - np.trace(similarity_matrix)) / (n_makhna * (n_makhna - 1)):.3f}")

## 8. Summary & Observations

In [ ]:
print("=" * 80)
print("FEATURE EXTRACTION EXPLORATION SUMMARY")
print("=" * 80)

print("\n📊 Model Architecture:")
print(f"  • Total parameters: {total_params:,}")
print(f"  • Feature dimension: 512")
print(f"  • Attention-based fusion: Enabled")

print("\n📸 Samples Processed:")
print(f"  • Makhna: {len(makhna_features)}")
print(f"  • Herd: {len(herd_features)}")

print("\n🎯 Key Findings:")
print(f"  • Features are L2-normalized (norm ≈ 1.0)")
print(f"  • Attention weights adapt per image")
print(f"  • Self-similarity = 1.0 (as expected)")

print("\n✅ Next Steps:")
print("  1. Prepare training data with identity labels")
print("  2. Implement metric learning loss (Triplet/ArcFace)")
print("  3. Train the model on full dataset")
print("  4. Evaluate on test set")

print("\n" + "=" * 80)

## Notes & Observations

*Add your observations here:*

- Attention weight patterns:
- Feature quality:
- Similarity patterns:
- Next experiments: